In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")
TMDB_ACCESS_TOKEN = os.getenv("TMDB_ACCESS_TOKEN")

In [2]:
headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_ACCESS_TOKEN}"
}

In [3]:
def search_tmdb_movie(title, year=None):
    url = "https://api.themoviedb.org/3/search/movie"

    params = {
        "query": title,
        "include_adult": False,
        "language": "en-US",
        "page": 1
    }

    if pd.notna(year):
        params["year"] = int(year)

    response = requests.get(url, headers=headers, params=params, timeout=15)

    if response.status_code != 200:
        return None

    data = response.json()
    results = data.get("results", [])

    if not results:
        return None

    return results[0]

In [4]:
result = search_tmdb_movie("The Dark Knight", 2008)
result

{'adult': False,
 'backdrop_path': '/cfT29Im5VDvjE0RpyKOSdCKZal7.jpg',
 'genre_ids': [28, 80, 53],
 'id': 155,
 'title': 'The Dark Knight',
 'original_language': 'en',
 'original_title': 'The Dark Knight',
 'overview': 'Batman raises the stakes in his war on crime. With the help of Lt. Jim Gordon and District Attorney Harvey Dent, Batman sets out to dismantle the remaining criminal organizations that plague the streets. The partnership proves to be effective, but they soon find themselves prey to a reign of chaos unleashed by a rising criminal mastermind known to the terrified citizens of Gotham as the Joker.',
 'popularity': 38.3873,
 'poster_path': '/qJ2tW6WMUDux911r6m7haRef0WH.jpg',
 'release_date': '2008-07-16',
 'softcore': False,
 'video': False,
 'vote_average': 8.53,
 'vote_count': 35691}

In [5]:
def get_tmdb_overview(row):
    result = search_tmdb_movie(
        row["primaryTitle"],
        row["startYear"]
    )

    if result is None:
        return pd.Series({
            "tmdb_id": None,
            "tmdb_title": None,
            "tmdb_release_date": None,
            "tmdb_overview": None
        })

    return pd.Series({
        "tmdb_id": result.get("id"),
        "tmdb_title": result.get("title"),
        "tmdb_release_date": result.get("release_date"),
        "tmdb_overview": result.get("overview")
    })

In [6]:
from pathlib import Path

ROOT = Path.cwd().parent
print(ROOT)
DATA_RAW = ROOT/"data/raw"
model_df = pd.read_csv(
    DATA_RAW/"default_model_w_franchise_n_plot.csv"
)

print(model_df.shape)
model_df.head()

c:\Users\sebas\PycharmProjects\Git\BoxOffice_Oracle
(2255, 50)


,Unnamed: 0,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,opening_weekend_gross,opening_theaters,domestic_release_date,...,actor_5,actor_6,director_name,writer_name,actor_1_name,actor_2_name,actor_3_name,plot_summary,has_plot_summary,franchise
0,0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,NaN,56061504.0,3602.0,2005-07-08,...,nm0573037,nm0512934,Tim Story,Mark Frost,Ioan Gruffudd,Michael Chiklis,Chris Evans,Scientist Reed Richards persuades his arrogant...,True,Fantastic Four
1,1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,NaN,19145480.0,3204.0,2005-09-16,...,nm0925768,NaN,Tim Burton,John August,Johnny Depp,Helena Bonham Carter,Emily Watson,NaN,False,NaN
2,2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,NaN,108435841.0,3661.0,2005-05-19,...,nm0001519,nm0001751,George Lucas,George Lucas,Hayden Christensen,Natalie Portman,Ewan McGregor,"Years after the onset of the Clone Wars, the n...",True,Star Wars
3,3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,NaN,23172440.0,3028.0,2004-04-02,...,nm1140344,nm0734558,Guillermo del Toro,Guillermo del Toro,Ron Perlman,Doug Jones,Selma Blair,NaN,False,Hellboy
4,4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,NaN,5935256.0,1603.0,2008-03-07,...,nm0269077,nm0202810,Roger Donaldson,Dick Clement,Jason Statham,Saffron Burrows,Stephen Campbell Moore,Self-reformed petty criminal Terry Leather has...,True,NaN


In [7]:
test_df = model_df.head(20).copy()

tmdb_test = test_df.apply(get_tmdb_overview, axis=1)

tmdb_test.head()

,tmdb_id,tmdb_title,tmdb_release_date,tmdb_overview
0,9738,Fantastic Four,2005-06-29,"During a space voyage, four scientists are alt..."
1,3933,Corpse Bride,2005-09-12,"In a 19th-century European village, a young ma..."
2,1895,Star Wars: Episode III - Revenge of the Sith,2005-05-17,When the sinister Sith unveil a thousand-year-...
3,1487,Hellboy,2004-04-02,"In the final days of World War II, the Nazis a..."
4,8848,The Bank Job,2008-02-28,Terry is a small-time car dealer trying to lea...


In [8]:
tmdb_rows = []

for i, row in model_df.iterrows():

    existing_plot = row.get("plot_summary")

    # Skip if already has synopsis
    if pd.notna(existing_plot) and str(existing_plot).strip() != "":
        tmdb_rows.append(pd.Series({
            "tmdb_id": None,
            "tmdb_title": None,
            "tmdb_release_date": None,
            "tmdb_overview": None,
            "tmdb_lookup_skipped": True
        }))

        continue

    tmdb_data = get_tmdb_overview(row)

    tmdb_data["tmdb_lookup_skipped"] = False

    tmdb_rows.append(tmdb_data)

    if i % 100 == 0:
        print(f"[{i}/{len(model_df)}] done")

    time.sleep(0.25)

[100/2255] done
[200/2255] done
[300/2255] done
[700/2255] done
[1000/2255] done


In [9]:
tmdb_df = pd.DataFrame(tmdb_rows)

In [10]:
model_df_tmdb = pd.concat(
    [model_df.reset_index(drop=True), tmdb_df.reset_index(drop=True)],
    axis=1
)

In [11]:
model_df_tmdb["final_market_synopsis"] = (
    model_df_tmdb["plot_summary"]
    .fillna(model_df_tmdb["tmdb_overview"])
)

In [12]:
model_df_tmdb.head()

,Unnamed: 0,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,opening_weekend_gross,opening_theaters,domestic_release_date,...,actor_3_name,plot_summary,has_plot_summary,franchise,tmdb_id,tmdb_title,tmdb_release_date,tmdb_overview,tmdb_lookup_skipped,final_market_synopsis
0,0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,NaN,56061504.0,3602.0,2005-07-08,...,Chris Evans,Scientist Reed Richards persuades his arrogant...,True,Fantastic Four,NaN,NaN,NaN,NaN,True,Scientist Reed Richards persuades his arrogant...
1,1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,NaN,19145480.0,3204.0,2005-09-16,...,Emily Watson,NaN,False,NaN,3933.0,Corpse Bride,2005-09-12,"In a 19th-century European village, a young ma...",False,"In a 19th-century European village, a young ma..."
2,2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,NaN,108435841.0,3661.0,2005-05-19,...,Ewan McGregor,"Years after the onset of the Clone Wars, the n...",True,Star Wars,NaN,NaN,NaN,NaN,True,"Years after the onset of the Clone Wars, the n..."
3,3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,NaN,23172440.0,3028.0,2004-04-02,...,Selma Blair,NaN,False,Hellboy,1487.0,Hellboy,2004-04-02,"In the final days of World War II, the Nazis a...",False,"In the final days of World War II, the Nazis a..."
4,4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,NaN,5935256.0,1603.0,2008-03-07,...,Stephen Campbell Moore,Self-reformed petty criminal Terry Leather has...,True,NaN,NaN,NaN,NaN,NaN,True,Self-reformed petty criminal Terry Leather has...


In [13]:
model_df_tmdb["final_market_synopsis"].isna().mean()

np.float64(0.0)

In [14]:
model_df_tmdb.columns

Index(['Unnamed: 0', 'tconst', 'primaryTitle', 'startYear', 'the_numbers_url',
       'scrape_success', 'scrape_error', 'opening_weekend_gross',
       'opening_theaters', 'domestic_release_date', 'release_type',
       'all_domestic_release_types', 'distributor', 'production_budget',
       'MPA_rating', 'MPA_rating_reason', 'runtime_minutes', 'genre',
       'subgenre', 'target_audience', 'time_period_setting', 'source',
       'production_method', 'creative_type', 'production_countries',
       'languages', 'legs', 'plot_point', 'raw_opening_weekend_text',
       'raw_theater_counts_text', 'raw_domestic_releases_text',
       'log_opening_weekend_gross', 'release_month', 'release_day_of_year',
       'director_id', 'writer_id', 'actor_1', 'actor_2', 'actor_3', 'actor_4',
       'actor_5', 'actor_6', 'director_name', 'writer_name', 'actor_1_name',
       'actor_2_name', 'actor_3_name', 'plot_summary', 'has_plot_summary',
       'franchise', 'tmdb_id', 'tmdb_title', 'tmdb_release_date

In [15]:
drop_cols = [
    "Unnamed: 0",
    "scrape_success",
    "scrape_error",
    "raw_opening_weekend_text",
    "raw_theater_counts_text",
    "raw_domestic_releases_text",
    "tmdb_lookup_skipped",
    "the_numbers_url",
    "tmdb_id",
    "tmdb_title",
    "tmdb_release_date",
    "tmdb_overview",
    "has_plot_summary",
    "plot_summary"
]
model_df_tmdb=model_df_tmdb.drop(columns=drop_cols)

In [16]:
model_df_tmdb.head()

,tconst,primaryTitle,startYear,opening_weekend_gross,opening_theaters,domestic_release_date,release_type,all_domestic_release_types,distributor,production_budget,...,actor_4,actor_5,actor_6,director_name,writer_name,actor_1_name,actor_2_name,actor_3_name,franchise,final_market_synopsis
0,tt0120667,Fantastic Four,2005.0,56061504.0,3602.0,2005-07-08,Wide,Wide,20th Century Fox,87500000.0,...,nm0004695,nm0573037,nm0512934,Tim Story,Mark Frost,Ioan Gruffudd,Michael Chiklis,Chris Evans,Fantastic Four,Scientist Reed Richards persuades his arrogant...
1,tt0121164,Corpse Bride,2005.0,19145480.0,3204.0,2005-09-16,Expands Wide,Expands Wide,Warner Bros.,40000000.0,...,nm0001808,nm0925768,NaN,Tim Burton,John August,Johnny Depp,Helena Bonham Carter,Emily Watson,NaN,"In a 19th-century European village, a young ma..."
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,108435841.0,3661.0,2005-05-19,Wide,Wide,20th Century Fox,115000000.0,...,nm0000168,nm0001519,nm0001751,George Lucas,George Lucas,Hayden Christensen,Natalie Portman,Ewan McGregor,Star Wars,"Years after the onset of the Clone Wars, the n..."
3,tt0167190,Hellboy,2004.0,23172440.0,3028.0,2004-04-02,Wide,Wide,Sony Pictures,60000000.0,...,nm0000457,nm1140344,nm0734558,Guillermo del Toro,Guillermo del Toro,Ron Perlman,Doug Jones,Selma Blair,Hellboy,"In the final days of World War II, the Nazis a..."
4,tt0200465,The Bank Job,2008.0,5935256.0,1603.0,2008-03-07,Wide,Wide,Lionsgate,20000000.0,...,nm0990547,nm0269077,nm0202810,Roger Donaldson,Dick Clement,Jason Statham,Saffron Burrows,Stephen Campbell Moore,NaN,Self-reformed petty criminal Terry Leather has...


In [17]:
model_df_tmdb.to_csv(
    DATA_RAW / "00_model_df.csv",
    index=False
)